# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This helps you understand the structure and find the relevant @id values to reference data and columns in later steps.

In [ ]:
# List available record sets and their fields by @id

record_sets = dataset.record_sets

print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name if hasattr(rs, 'name') else rs.id}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    # List fields
    if hasattr(rs, 'fields'):
        print(f"  Fields and columns:")
        for f in rs.fields:
            print(f"    - Field: {f.name} (@id: {f.id})  | DataType: {f.data_type}")
            if hasattr(f, 'columns') and f.columns is not None:
                for c in f.columns:
                    print(f"      > Column: {c.name} (@id: {c.id})")
    print("\n------------------\n")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above. This example loads **ALL** record sets and prints the columns for one example record set.

**NOTE**: For this dataset, let's use the first main record set, which is typically the most relevant table.

In [ ]:
# Choose the record_set @id by inspecting the overview above; in FAIR^2, typically a main record set table is present.

# List of available record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"    Columns: {df.columns.tolist()}")
    print(f"    First 2 rows:\n{df.head(2)}\n")

# For following cells, pick the first record_set_id
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Using main record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**NOTE:** Replace `<numeric_field_id>` and `<group_field_id>` with the appropriate `@id` from the fields in your selected record set. You can get these from the overview above or from the DataFrame columns.

In [ ]:
# Inspect column names and pick common fields for filtering/EDA
df = dataframes[main_record_set_id]
print(f"Columns in main record set DataFrame:\n{df.columns.tolist()}")

# Example: Use a numeric field such as 'cr:field:normalized_abundance' for filtering/normalizing
# and another field such as 'cr:field:sample' as grouping
numeric_field = None
group_field = None

# Try to auto-pick likely numeric/group columns
for c in df.columns:
    if numeric_field is None and ("abundance" in c.lower() or "coverage" in c.lower() or "mw" in c.lower() or "counts" in c.lower()):
        numeric_field = c
    if group_field is None and ("sample" in c.lower() or "type" in c.lower()):
        group_field = c

print(f"Selected numeric_field: {numeric_field}")
print(f"Selected group_field: {group_field}")

# Drop records with missing or non-numeric values
filtered_df = df.copy()
if numeric_field and numeric_field in filtered_df.columns:
    # Convert strings to numeric if required
    filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')
    threshold = filtered_df[numeric_field].mean()  # Example: filter above the mean
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    print(filtered_df[[numeric_field]].head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below are examples using `matplotlib` to plot the distribution of the selected numeric field and the grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

# Barplot of grouped means (if available)
if group_field and numeric_field and group_field in df.columns and numeric_field in df.columns:
    group_df = df[[group_field, numeric_field]].copy()
    group_df[numeric_field] = pd.to_numeric(group_df[numeric_field], errors='coerce')
    means = group_df.groupby(group_field)[numeric_field].mean().dropna()
    means = means.sort_values(ascending=False)

    plt.figure(figsize=(8,4))
    sns.barplot(x=means.index, y=means.values)
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
We explored the FAIR<sup>2</sup> dataset on mass spectrometry analysis of extracellular vesicles from human mast cells. Using its Croissant schema, we:

- Inspected metadata and available record sets and fields (referenced via their `@id`)
- Loaded and examined the main data records as a pandas DataFrame
- Applied basic filtering and normalization to a numeric field
- Grouped data by a categorical field for comparison
- Visualized key distributions

Continue your exploration by applying further statistical analysis, feature engineering, or integrating this pipeline with your data science or ML workflow!